# Phase 6 DAEAC MCC FCBA RR Late-Fusion NSV: All Domain Adaptation Scenarios

Runs MCC adaptation while preserving the FCBA + RR late-fusion NSV architecture for DS1->DS2, DS1->INCART, DS1->SVDB, MITBIH->INCART, and MITBIH->SVDB.

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys
import numpy as np
import pandas as pd
os.environ['PYTHONUNBUFFERED'] = '1'

REPO_URL = 'https://github.com/YOUR_ORG/Best-thesis-in-council.git'
BRANCH = 'main'

SCENARIOS = [
    ('ds1_ds2', 'configs/phase6_daeac_fcba_latefusion_rr_nsv_mcc_ds1_ds2.yaml', 'daeac_fcba_rr_nsv_mcc_ds1_ds2'),
    ('ds1_incart', 'configs/phase6_daeac_fcba_latefusion_rr_nsv_mcc_ds1_incart.yaml', 'daeac_fcba_rr_nsv_mcc_ds1_incart'),
    ('ds1_svdb', 'configs/phase6_daeac_fcba_latefusion_rr_nsv_mcc_ds1_svdb.yaml', 'daeac_fcba_rr_nsv_mcc_ds1_svdb'),
    ('mitbih_incart', 'configs/phase6_daeac_fcba_latefusion_rr_nsv_mcc_mitbih_incart.yaml', 'daeac_fcba_rr_nsv_mcc_mitbih_incart'),
    ('mitbih_svdb', 'configs/phase6_daeac_fcba_latefusion_rr_nsv_mcc_mitbih_svdb.yaml', 'daeac_fcba_rr_nsv_mcc_mitbih_svdb'),
]

WORK = Path('/kaggle/working')
REPO = WORK / 'Best-thesis-in-council'
ECG = REPO / 'ecg_thesis'
print('Repo path:', ECG)

## 1. Clone Repo and Install

In [ ]:
if not ECG.exists():
    assert 'YOUR_ORG' not in REPO_URL, 'Edit REPO_URL before cloning.'
    !git clone --branch {BRANCH} {REPO_URL} /kaggle/working/Best-thesis-in-council
else:
    print('Repo already exists:', ECG)
assert ECG.exists(), ECG
os.chdir(ECG)
!pip install -q -r requirements.txt
!python scripts/check_repo.py

## 2. Copy NSV RR Bundle

In [ ]:
candidate_dirs = [p.parent for p in Path('/kaggle/input').glob('**/record_split_manifest.json')]
print('Candidate record-split bundle dirs:')
for p in candidate_dirs:
    print(' -', p)
assert len(candidate_dirs) == 1, f'Expected one NSV bundle, found {candidate_dirs}'

DEST = Path('data/processed/phase6_daeac_record_splits_nsv')
DEST.mkdir(parents=True, exist_ok=True)
for src in sorted(candidate_dirs[0].glob('*.npz')):
    shutil.copy2(src, DEST / src.name)
manifest = candidate_dirs[0] / 'record_split_manifest.json'
if manifest.exists():
    shutil.copy2(manifest, DEST / manifest.name)

required_npz = [
    'ds1_train.npz', 'ds1_val.npz', 'mitbih_train.npz', 'mitbih_val.npz',
    'ds2_train.npz', 'ds2_val.npz', 'ds2_test.npz',
    'incart_train.npz', 'incart_val.npz', 'incart_test.npz',
    'svdb_train.npz', 'svdb_val.npz', 'svdb_test.npz',
]
for name in required_npz:
    path = DEST / name
    assert path.exists(), f'Missing {path}'
    with np.load(path, allow_pickle=True) as data:
        assert data['class_names'].tolist() == ['N', 'S', 'V'], (name, data['class_names'].tolist())
        assert 'rr_features' in data.files, f'{name} has no rr_features.'
        assert data['rr_features'].shape == (len(data['x']), 7), (name, data['rr_features'].shape)

print('Copied NSV bundle to', DEST)
!ls -lh data/processed/phase6_daeac_record_splits_nsv | head

## 3. Copy Required Pretrain Checkpoints

In [ ]:
CHECKPOINTS = {
    'daeac_fcba_rr_nsv_sqrtw_base_best.pt': Path('outputs/phase6_daeac_fcba_latefusion_rr_nsv_sqrtw/checkpoints/daeac_fcba_rr_nsv_sqrtw_base_best.pt'),
    'daeac_fcba_rr_nsv_mitbih_sqrtw_base_best.pt': Path('outputs/phase6_daeac_fcba_latefusion_rr_nsv_mitbih_sqrtw/checkpoints/daeac_fcba_rr_nsv_mitbih_sqrtw_base_best.pt'),
}

for filename, dest in CHECKPOINTS.items():
    candidates = sorted(Path('/kaggle/input').glob(f'**/{filename}'))
    print(filename, 'candidates:', candidates)
    assert len(candidates) == 1, f'Expected exactly one {filename}, found {candidates}'
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(candidates[0], dest)
    print('copied', candidates[0], '->', dest)

for dest in CHECKPOINTS.values():
    assert dest.exists(), dest
!find outputs -path '*checkpoints*best.pt' -maxdepth 5 -type f -print

## 4. Validate All Configs

In [ ]:
for scenario, config, _ in SCENARIOS:
    print('\nVALIDATE', scenario)
    subprocess.run([sys.executable, 'scripts/phase6_daeac_paper/00_validate_data.py', '--config', config], check=True)

## 5. Run MCC Adaptation and Evaluate Target Test

In [ ]:
rows = []
for scenario, config, prefix in SCENARIOS:
    print('\n' + '=' * 88)
    print('MCC ADAPT', scenario)
    print('=' * 88)
    subprocess.run([
        sys.executable, '-u', 'scripts/phase6_daeac_mcc/01_train.py',
        '--config', config,
    ], check=True)

    train_summary_candidates = sorted(Path('outputs').glob(f'**/metrics/{prefix}_train_summary.json'))
    train_summary_candidates = [p for p in train_summary_candidates if scenario in str(p)]
    assert len(train_summary_candidates) == 1, (scenario, train_summary_candidates)
    adapt_summary_path = train_summary_candidates[0]
    adapt_summary = json.loads(adapt_summary_path.read_text())
    best_checkpoint = Path(adapt_summary['best_checkpoint'])
    assert best_checkpoint.exists(), best_checkpoint

    method = f'{prefix}_best'
    subprocess.run([
        sys.executable, '-u', 'scripts/phase6_daeac_paper/03_eval.py',
        '--config', config,
        '--checkpoint', str(best_checkpoint),
        '--method-name', method,
        '--dataset', 'target',
    ], check=True)

    metrics_path = adapt_summary_path.parents[1] / 'metrics' / f'{method}_target_test_metrics.json'
    cm_path = adapt_summary_path.parents[1] / 'metrics' / f'{method}_target_test_confusion_matrix.csv'
    assert metrics_path.exists(), metrics_path
    metrics = json.loads(metrics_path.read_text())
    rows.append({
        'scenario': scenario,
        'best_epoch': adapt_summary.get('best_epoch'),
        'best_v_measure': adapt_summary.get('best_v_measure'),
        'target_accuracy': metrics['accuracy'],
        'target_macro_f1': metrics['macro_f1'],
        'N_f1': metrics['per_class']['N']['f1'],
        'S_f1': metrics['per_class']['S']['f1'],
        'V_f1': metrics['per_class']['V']['f1'],
        'best_checkpoint': str(best_checkpoint),
        'metrics_path': str(metrics_path),
        'confusion_matrix_path': str(cm_path),
    })

df = pd.DataFrame(rows).sort_values('scenario')
display(df)
summary_csv = Path('outputs/phase6_daeac_fcba_latefusion_rr_nsv_mcc_all_domains_summary.csv')
summary_csv.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(summary_csv, index=False)
print('Saved', summary_csv)

## 6. Package Outputs

In [ ]:
!zip -r /kaggle/working/phase6_daeac_fcba_latefusion_rr_nsv_mcc_all_domains_outputs.zip outputs/phase6_daeac_fcba_latefusion_rr_nsv_mcc_* outputs/phase6_daeac_fcba_latefusion_rr_nsv_mcc_all_domains_summary.csv